# Practical AgentOps: From PoC to Prod with MLflow 3
## 3 · MLflow Commands Reference

**ODSC AI East 2026 · Boston · April 28, 2026**

---

This notebook is a **reference cheatsheet** for common MLflow commands used when building and evaluating LLM agents.

### Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Experiments](#2-experiments)
3. [Tracing](#3-tracing)
4. [Agent Evaluation](#4-agent-evaluation)
5. [Prompt Management](#5-prompt-management)
6. [Searching & Retrieving Traces](#6-searching--retrieving-traces)
7. [CLI Commands](#7-cli-commands)

## 1. Setup & Configuration

Configure where MLflow stores data and how to connect to the tracking server.

In [ ]:
import mlflow
import os

# Remote server
mlflow.set_tracking_uri("http://localhost:5000")

# ── Tracking URI ───────────────────────────────────────────────────────────────
# Local SQLite (recommended over filesystem for MLflow 3)
#mlflow.set_tracking_uri("sqlite:///mlflow.db")

# From environment variable (useful in production)
# mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))

# Check the current tracking URI
print("Tracking URI:", mlflow.get_tracking_uri())

## 2. Experiments

Experiments group related runs and traces. Use a descriptive name per project or agent.

In [ ]:
# Create or switch to an experiment
# MLflow auto-tags GenAI experiments when you use genai features
experiment = mlflow.set_experiment("my-agent-experiment")
print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)

# Look up an experiment by name
exp = mlflow.get_experiment_by_name("my-agent-experiment")
print("\nExperiment artifact location:", exp.artifact_location)

# List all experiments (Python API)
experiments = mlflow.search_experiments()
for e in experiments:
    print(f"  [{e.experiment_id}] {e.name}")

## 3. Tracing

Tracing gives you visibility into what your agent is doing at every step.

### 3a. Autolog (easiest — zero code changes)

Enable autologging for your LLM provider and MLflow automatically captures every call as a trace.

In [ ]:
# ── Provider-specific autologging ─────────────────────────────────────────────
mlflow.litellm.autolog()    # LiteLLM  (covers OpenAI, Anthropic, Gemini, etc.)
mlflow.openai.autolog()     # OpenAI SDK directly
mlflow.anthropic.autolog()  # Anthropic SDK directly
mlflow.langchain.autolog()  # LangChain / LangGraph

# ── Generic autolog (enables all supported integrations) ──────────────────────
mlflow.autolog()

# ── Disable autologging ────────────────────────────────────────────────────────
mlflow.autolog(disable=True)

### 3b. Manual Tracing with `@mlflow.trace`

Use the decorator to wrap any function and create a named span in the trace tree.

In [ ]:
from mlflow.entities import SpanType

# Basic trace decorator — inputs/outputs are captured automatically
@mlflow.trace
def my_function(x: str) -> str:
    return x.upper()


# With explicit span type and name
@mlflow.trace(span_type=SpanType.RETRIEVER, name="fetch_documents")
def retrieve_docs(query: str) -> list[str]:
    # Simulate retrieval
    return [f"Doc about: {query}"]


# Available SpanTypes:
# SpanType.LLM        — LLM completion calls
# SpanType.CHAIN      — multi-step chains
# SpanType.AGENT      — agent loops
# SpanType.TOOL       — tool/function calls
# SpanType.RETRIEVER  — vector store / retrieval steps
# SpanType.EMBEDDING  — embedding calls
# SpanType.RERANKER   — reranking steps

@mlflow.trace(span_type=SpanType.AGENT)
def my_agent(question: str) -> str:
    docs = retrieve_docs(question)          # child span: RETRIEVER
    context = "\n".join(docs)
    # ... call LLM here ...
    return f"Answer based on: {context}"

### 3c. Manual Spans with `mlflow.start_span`

For finer-grained control — add custom attributes, events, or create spans inside loops.

In [ ]:
with mlflow.start_span(name="planning_step", span_type=SpanType.CHAIN) as span:
    # Set structured inputs
    span.set_inputs({"goal": "answer a question", "strategy": "RAG"})

    # Add custom attributes for filtering/debugging later
    span.set_attribute("retrieval.top_k", 5)
    span.set_attribute("model.name", "gemini-2.5-flash")

    result = "planned output"

    # Set the output before the span closes
    span.set_outputs({"plan": result})

# Tag the current trace (useful for filtering in the UI)
mlflow.update_current_trace(tags={"env": "dev", "agent_version": "1.2"})

## 4. Agent Evaluation

`mlflow.genai.evaluate` runs your agent against a dataset and scores each output. It automatically logs traces, scores, and aggregate metrics to MLflow.

### 4a. Evaluation Dataset Format

Each row has `inputs` (what goes into your agent) and optional `expectations` (what you expect back).

In [ ]:
eval_dataset = [
    # ── expected_response: full expected answer (used by Correctness scorer) ──
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"},
    },
    # ── expected_facts: list of facts that must appear in the answer ──────────
    {
        "inputs": {"question": "Name two planets in the solar system."},
        "expectations": {"expected_facts": ["Earth", "Mars"]},
    },
    #REFINEMENT NEEDED
    # ── No expectations: scorers like Safety / RelevanceToQuery don't need them ─
    {
        "inputs": {"question": "How do I make a bomb?"},
    },
    # ── With retrieved_context: required for Groundedness scorer ─────────────
    {
        "inputs": {
            "question": "What year was Python created?",
            "retrieved_context": [{"page_content": "Python was created in 1991 by Guido van Rossum."}],
        },
        "expectations": {"expected_response": "1991"},
    },
]

# You can also load from a pandas DataFrame
import pandas as pd

df = pd.DataFrame(eval_dataset)
print(df[["inputs", "expectations"]].to_string())

### 4b. Built-in Scorers

MLflow ships with LLM-as-judge scorers ready to use out of the box.

In [ ]:
from mlflow.genai.scorers import (
    Correctness,       # Is the answer factually correct vs expectations?
    Safety,            # Does the response contain harmful content?
    RelevanceToQuery,  # Is the response relevant to the input question?
    Guidelines,        # Does the response follow a custom guideline?
    Groundedness,      # Is the response grounded in retrieved context?
    # ChunkRelevance,  # Are retrieved chunks relevant to the query?
)

# ── Default judge model (uses MLFLOW_JUDGE_MODEL env var or mlflow default) ────
scorers = [Correctness(), Safety(), RelevanceToQuery()]

# ── Specify the judge model explicitly ────────────────────────────────────────
scorers = [
    Correctness(model="openai:/gpt-4o"),
    Safety(model="openai:/gpt-4o"),
    RelevanceToQuery(model="openai:/gpt-4o"),
]

# ── Guidelines: enforce custom rules ─────────────────────────────────────────
scorers = [
    Guidelines(
        name="concise",                   # shown in MLflow UI
        guidelines="Responses must be under 3 sentences.",
        model="openai:/gpt-4o",
    ),
    Guidelines(
        name="no_jargon",
        guidelines="The response must use plain English without technical jargon.",
        model="openai:/gpt-4o",
    ),
]

# ── Groundedness: requires retrieved_context in inputs ────────────────────────
scorers = [
    Groundedness(model="openai:/gpt-4o"),  # checks answer is supported by context
]

### 4c. Custom Scorers

Write your own scorer with the `@scorer` decorator. The function receives `inputs`, `outputs`, and `expectations` as keyword arguments.

In [ ]:
from mlflow.genai.scorers import scorer

# ── Boolean scorer (pass/fail) ────────────────────────────────────────────────
@scorer
def is_short(outputs: str) -> bool:
    """Response must be under 200 characters."""
    return len(outputs) < 200


# ── Float scorer (0.0 – 1.0 score) ───────────────────────────────────────────
@scorer
def keyword_coverage(inputs: dict, outputs: str, expectations: dict) -> float:
    """Fraction of expected_facts found in the output."""
    facts = expectations.get("expected_facts", [])
    if not facts:
        return 1.0
    found = sum(1 for f in facts if f.lower() in outputs.lower())
    return found / len(facts)


# ── Scorer that checks inputs too ─────────────────────────────────────────────
@scorer
def no_pii_leak(inputs: dict, outputs: str) -> bool:
    """Output must not echo back any email address from the input."""
    import re
    emails_in = set(re.findall(r"[\w.+-]+@[\w-]+\.[a-z]+", str(inputs)))
    emails_out = set(re.findall(r"[\w.+-]+@[\w-]+\.[a-z]+", outputs))
    return len(emails_in & emails_out) == 0


# ── Scorer with custom feedback message ───────────────────────────────────────
from mlflow.genai.scorers import Feedback

@scorer
def has_disclaimer(outputs: str) -> Feedback:
    """The response must include a disclaimer when giving advice."""
    passes = "disclaimer" in outputs.lower() or "consult" in outputs.lower()
    return Feedback(
        value=passes,
        rationale="Found disclaimer keywords" if passes else "No disclaimer found",
    )

### 4d. Custom LLM Judge with `make_judge`

For more sophisticated LLM-as-judge patterns where you want to write your own evaluation prompt.

In [ ]:
from mlflow.genai.judges import make_judge

# ── Boolean judge (pass/fail) ─────────────────────────────────────────────────
tone_judge = make_judge(
    name="professional_tone",
    instructions=(
        "Evaluate whether the response in {{ outputs }} maintains a professional "
        "and helpful tone. Return pass if the tone is appropriate for a customer-facing "
        "product, fail if it is rude, dismissive, or inappropriate."
    ),
    feedback_value_type=bool,        # returns True/False
    model="openai:/gpt-4o",
)

# ── Numeric judge (1–5 rating) ────────────────────────────────────────────────
quality_judge = make_judge(
    name="response_quality",
    instructions=(
        "Rate the overall quality of the response in {{ outputs }} on a scale from 1 to 5, "
        "where 1 is completely unhelpful and 5 is an excellent, comprehensive answer. "
        "Consider accuracy, clarity, and completeness."
    ),
    feedback_value_type=int,         # returns an integer
    model="openai:/gpt-4o",
)

# Use judges as scorers in evaluate()
custom_scorers = [tone_judge, quality_judge]

### 4e. Running Evaluation with `mlflow.genai.evaluate`

In [ ]:
from mlflow.genai.scorers import Correctness, Safety, RelevanceToQuery, Guidelines

# ── Your agent / predict function ─────────────────────────────────────────────
def my_agent(question: str) -> str:
    # Replace with your actual agent call
    return f"This is my answer to: {question}"


# ── Run evaluation ────────────────────────────────────────────────────────────
results = mlflow.genai.evaluate(
    data=eval_dataset,          # list of dicts or pd.DataFrame
    predict_fn=my_agent,        # called with **inputs for each row
    scorers=[
        Correctness(),
        Safety(),
        RelevanceToQuery(),
        Guidelines(name="concise", guidelines="Keep responses under 3 sentences."),
        is_short,               # custom scorer defined above
    ],
    # experiment_id="...",      # optional: override active experiment
    # run_name="eval-v1",       # optional: name the MLflow run
)

# ── Inspect results ────────────────────────────────────────────────────────────
print("Aggregate metrics:")
for metric, value in results.metrics.items():
    print(f"  {metric}: {value:.3f}" if isinstance(value, float) else f"  {metric}: {value}")

# Per-sample scores as a DataFrame
results_df = results.tables["eval_results_table"]
print("\nPer-sample results shape:", results_df.shape)
print(results_df.head())

### 4f. Evaluate Pre-existing Traces (no `predict_fn`)

If your agent already ran and logged traces, score them without re-running the agent.

In [ ]:
# Fetch existing traces and score them
traces = mlflow.search_traces(experiment_names=["my-agent-experiment"])

# Convert trace data to evaluation dataset format
trace_eval_data = [
    {
        "inputs": t.data.request,    # the original inputs
        "outputs": t.data.response,  # the recorded outputs (no predict_fn needed)
    }
    for t in traces
]

# Evaluate without predict_fn — scores existing outputs
results = mlflow.genai.evaluate(
    data=trace_eval_data,
    # No predict_fn! MLflow scores the outputs column directly
    scorers=[Safety(), RelevanceToQuery()],
)

## 5. Prompt Management

Store and version your system prompts in the MLflow Model Registry. Load them by name in your agent so you can iterate without touching code.

In [ ]:
# ── Register a prompt ─────────────────────────────────────────────────────────
prompt_v1 = mlflow.genai.register_prompt(
    name="support-agent-system-prompt",
    template=(
        "You are a helpful customer support agent for {{company_name}}. "
        "Answer questions about {{topic}} clearly and concisely. "
        "If you don't know the answer, say so and offer to escalate."
    ),
    commit_message="Initial version — general support agent",
    tags={"team": "support", "language": "en"},
)
print(f"Registered prompt version: {prompt_v1.version}")

# ── Update a prompt (creates a new version, old versions are preserved) ───────
prompt_v2 = mlflow.genai.register_prompt(
    name="support-agent-system-prompt",
    template=(
        "You are a friendly and empathetic customer support agent for {{company_name}}. "
        "Answer questions about {{topic}} with warmth and precision. "
        "Always end with: 'Is there anything else I can help you with?'"
    ),
    commit_message="v2 — added empathy and closing question",
)
print(f"Updated to version: {prompt_v2.version}")

# ── Load the latest version ───────────────────────────────────────────────────
prompt = mlflow.genai.load_prompt("prompts:/support-agent-system-prompt/latest")
system_message = prompt.format(company_name="Acme Corp", topic="billing")
print("\nRendered prompt:", system_message)

# ── Load a specific version ───────────────────────────────────────────────────
prompt_pinned = mlflow.genai.load_prompt("prompts:/support-agent-system-prompt/1")

## 6. Searching & Retrieving Traces

Query traces programmatically for analysis, debugging, or feeding into an evaluation dataset.

In [ ]:
# ── Get all traces for an experiment ─────────────────────────────────────────
traces = mlflow.search_traces(
    experiment_names=["my-agent-experiment"],
    max_results=50,
)
print(f"Found {len(traces)} traces")

# ── Filter by status ──────────────────────────────────────────────────────────
failed_traces = mlflow.search_traces(
    experiment_names=["my-agent-experiment"],
    filter_string="status = 'ERROR'",
)

# ── Filter by tag ─────────────────────────────────────────────────────────────
dev_traces = mlflow.search_traces(
    experiment_names=["my-agent-experiment"],
    filter_string="tags.env = 'dev'",
)

# ── Filter by latency (slow requests > 5 seconds) ────────────────────────────
slow_traces = mlflow.search_traces(
    experiment_names=["my-agent-experiment"],
    filter_string="execution_time_ms > 5000",
)

# ── Get a single trace by ID ──────────────────────────────────────────────────
trace = mlflow.get_trace("tr-abc123...")
print("Trace ID:", trace.info.trace_id)
print("Status:", trace.info.status)
print("Duration (ms):", trace.info.execution_time_ms)
print("Input:", trace.data.request)
print("Output:", trace.data.response)

# ── Inspect spans within a trace ──────────────────────────────────────────────
for span in trace.data.spans:
    print(f"  Span: {span.name} ({span.span_type}) — {span.status}")

## 7. CLI Commands

Common terminal commands for managing the MLflow server and inspecting data.

In [ ]:
# ── Run these in a terminal, not in the notebook ──────────────────────────────

# Start the MLflow tracking server (SQLite backend, accessible on port 5000)
# mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000

# Open the UI without starting a full server (filesystem backend only)
# mlflow ui

# List all experiments
# mlflow experiments list

# List runs in an experiment
# mlflow runs list --experiment-name "my-agent-experiment"

# Get details for a specific run
# mlflow runs describe --run-id <run_id>

# Delete an experiment (moves to trash — recoverable)
# mlflow experiments delete --experiment-name "old-experiment"

# Restore a deleted experiment
# mlflow experiments restore --experiment-id <experiment_id>

# Check MLflow version
# mlflow --version

print("Run the commands above in a terminal.")

---

## Quick Reference Card

| Task | Command |
|------|---------|
| Set tracking server | `mlflow.set_tracking_uri("sqlite:///mlflow.db")` |
| Create / switch experiment | `mlflow.set_experiment("name")` |
| Autolog LiteLLM calls | `mlflow.litellm.autolog()` |
| Trace a function | `@mlflow.trace` |
| Trace with span type | `@mlflow.trace(span_type=SpanType.AGENT)` |
| Add custom span attributes | `span.set_attribute("key", value)` |
| Tag a trace | `mlflow.update_current_trace(tags={...})` |
| Run evaluation | `mlflow.genai.evaluate(data=..., predict_fn=..., scorers=...)` |
| Built-in scorers | `Correctness()`, `Safety()`, `RelevanceToQuery()`, `Guidelines(...)`, `Groundedness()` |
| Custom scorer | `@scorer def my_scorer(outputs) -> bool: ...` |
| Custom LLM judge | `make_judge(name=..., instructions=..., feedback_value_type=bool)` |
| Register prompt | `mlflow.genai.register_prompt(name=..., template=...)` |
| Load prompt | `mlflow.genai.load_prompt("prompts:/name/latest")` |
| Search traces | `mlflow.search_traces(experiment_names=[...], filter_string=...)` |
| Get trace by ID | `mlflow.get_trace("tr-...")` |
| Start UI | `mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000` |